# App User Behavior Segmentation
Run cells in order. Without a CSV this notebook uses synthetic demonstration data, not the original Google Drive dataset.

## Cell 1: Install libraries

In [ ]:
%pip install numpy pandas matplotlib seaborn scikit-learn joblib threadpoolctl

## Cell 2: Imports and configuration

In [ ]:
from pathlib import Path
import json, joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
from sklearn.metrics import silhouette_score, davies_bouldin_score, calinski_harabasz_score, adjusted_rand_score
from threadpoolctl import threadpool_limits
DATA_PATH = None  # Replace with your CSV filename for actual data.
COLUMN_MAPPING = {}
FEATURES = ['sessions_per_week','avg_session_minutes','active_days_30','features_used','days_since_last_login']
OUT = Path('demo_results' if DATA_PATH is None else 'actual_results')
OUT.mkdir(exist_ok=True, parents=True)

## Cell 3: Synthetic demonstration generator

In [ ]:
def demo_data(n=50000):
    rng=np.random.default_rng(42)
    groups=rng.choice(4,n,p=[.25,.35,.25,.15])
    centers=np.array([[26,32,26,11,1],[12,17,17,7,4],[2,4,3,2,23],[4,26,6,5,10]])
    x=np.maximum(centers[groups]+rng.normal(size=(n,5))*[2.5,3,2,1,1.8],0)
    x[:,2]=np.clip(np.round(x[:,2]),0,30)
    x[:,3]=np.round(x[:,3])
    result=pd.DataFrame(x,columns=FEATURES)
    result.insert(0,'user_id',[f'USER_{i:06d}' for i in range(n)])
    for c in FEATURES:
        result.loc[rng.choice(n,int(n*.01),replace=False),c]=np.nan
    return pd.concat([result,result.iloc[:50]],ignore_index=True)

## Cell 4: Load data

In [ ]:
raw=demo_data() if DATA_PATH is None else pd.read_csv(DATA_PATH).rename(columns=COLUMN_MAPPING)
if raw.columns.duplicated().any(): raise ValueError('Duplicate columns after mapping')
print('SYNTHETIC DEMONSTRATION' if DATA_PATH is None else DATA_PATH)
display(raw.head())
raw.info()

## Cell 5: Validate users and clean features

In [ ]:
required=['user_id']+FEATURES
missing=set(required)-set(raw.columns)
if missing: raise ValueError(f'Missing columns: {sorted(missing)}')
df=raw.drop_duplicates().reset_index(drop=True).copy()
df.user_id=df.user_id.astype('string').str.strip()
if df.user_id.isna().any() or df.user_id.eq('').any() or df.user_id.duplicated().any():
    raise ValueError('Require one row per unique nonmissing user_id')
def clean_features(data):
    x=data[FEATURES].apply(pd.to_numeric,errors='coerce').replace([np.inf,-np.inf],np.nan)
    x=x.mask(x<0)
    x.loc[x.active_days_30>30,'active_days_30']=np.nan
    return x
X=clean_features(df)
if len(X)<20 or X.isna().all().any(): raise ValueError('Insufficient users or entirely missing feature')
quality={'synthetic_demo':DATA_PATH is None,'users':len(df),'duplicates_removed':len(raw)-len(df),'missing_after_cleaning':X.isna().sum().to_dict()}
(OUT/'data_quality.json').write_text(json.dumps(quality,indent=2))
display(X.describe())

## Cell 6: EDA distributions

In [ ]:
X.describe().to_csv(OUT/'eda_summary.csv')
fig,axes=plt.subplots(2,3,figsize=(14,8))
for ax,c in zip(axes.flat,FEATURES):
    ax.hist(X[c].dropna(),bins=35); ax.set_title(c)
axes.flat[-1].axis('off')
fig.tight_layout(); fig.savefig(OUT/'eda_distributions.png'); plt.show()

## Cell 7: Correlation heatmap

In [ ]:
fig,ax=plt.subplots(figsize=(10,7))
sns.heatmap(X.corr(),annot=True,cmap='coolwarm',center=0,ax=ax)
fig.tight_layout(); fig.savefig(OUT/'eda_correlation.png'); plt.show()

## Cell 8: Imputation and scaling

In [ ]:
prep=Pipeline([('imputer',SimpleImputer(strategy='median')),('scaler',StandardScaler())])
Z=prep.fit_transform(X)
if len(np.unique(Z,axis=0))<9: raise ValueError('Need at least 9 distinct behavioral records')
display(pd.DataFrame(Z,columns=FEATURES).head())

## Cell 9: Evaluate candidate cluster counts

In [ ]:
models={}; rows=[]
with threadpool_limits(limits=2):
    for k in range(2,9):
        m=KMeans(n_clusters=k,n_init=10,random_state=42).fit(Z)
        models[k]=m
        rows.append({'k':k,'inertia':m.inertia_,'silhouette':silhouette_score(Z,m.labels_,sample_size=min(1500,len(Z)),random_state=42),'davies_bouldin':davies_bouldin_score(Z,m.labels_),'calinski_harabasz':calinski_harabasz_score(Z,m.labels_)})
evaluation=pd.DataFrame(rows)
evaluation.to_csv(OUT/'cluster_evaluation.csv',index=False)
display(evaluation)

## Cell 10: Select a model

In [ ]:
REQUESTED_K=None  # Set to 4 for an explicit four-cluster comparison.
best=int(evaluation.loc[evaluation.silhouette.idxmax(),'k'])
selected=best if REQUESTED_K is None else REQUESTED_K
if selected not in models: raise ValueError('k must be 2..8')
model=models[selected]
df['cluster_id']=model.labels_
print('Selected k:',selected)

## Cell 11: Elbow and silhouette plots

In [ ]:
fig,axes=plt.subplots(1,2,figsize=(13,5))
axes[0].plot(evaluation.k,evaluation.inertia,'o-'); axes[0].set(title='Elbow diagnostic',xlabel='k',ylabel='Inertia')
axes[1].plot(evaluation.k,evaluation.silhouette,'o-'); axes[1].axvline(selected,color='red',ls='--'); axes[1].set(title='Sampled silhouette',xlabel='k')
fig.tight_layout(); fig.savefig(OUT/'cluster_selection.png'); plt.show()

## Cell 12: Profiles and segment names

In [ ]:
imputed=pd.DataFrame(prep.named_steps['imputer'].transform(X),columns=FEATURES)
imputed['cluster_id']=model.labels_
profiles=imputed.groupby('cluster_id')[FEATURES].mean()
profiles['users']=df.groupby('cluster_id').size()
profiles['percentage']=profiles.users/len(df)*100
centers=pd.DataFrame(model.cluster_centers_,columns=FEATURES)
engagement=centers.mul([1,1,1,1,-1],axis=1).mean(axis=1)
order=engagement.sort_values().index.tolist()
names={c:f'Engagement tier {i+1} of {selected}' for i,c in enumerate(order)}
names[order[0]]='Low engagement - review retention'
names[order[-1]]='High engagement'
df['segment']=df.cluster_id.map(names)
profiles['segment']=profiles.index.map(names)
profiles['engagement_index']=engagement
profiles['suggested_action']=['Test reactivation and review onboarding' if c==order[0] else 'Test loyalty and advanced features' if c==order[-1] else 'Test relevant feature education' for c in profiles.index]
display(profiles)

## Cell 13: PCA visualization

In [ ]:
pca=PCA(n_components=2,random_state=42)
coords=pca.fit_transform(Z)
pca_df=pd.DataFrame({'user_id':df.user_id,'PC1':coords[:,0],'PC2':coords[:,1],'cluster_id':model.labels_})
sample=pca_df.sample(min(10000,len(df)),random_state=42)
fig,ax=plt.subplots(figsize=(11,7))
sns.scatterplot(data=sample,x='PC1',y='PC2',hue='cluster_id',palette='tab10',s=10,alpha=.4,ax=ax)
ax.set_title('PCA visualization — not independent validation')
fig.tight_layout(); fig.savefig(OUT/'pca_clusters.png'); plt.show()
print('Two-component explained variance:',pca.explained_variance_ratio_.sum())

## Cell 14: Cluster centers and sizes

In [ ]:
fig,ax=plt.subplots(figsize=(11,6))
sns.heatmap(centers,annot=True,center=0,cmap='vlag',ax=ax)
fig.tight_layout(); fig.savefig(OUT/'cluster_centers_heatmap.png'); plt.show()
fig,ax=plt.subplots(figsize=(9,5))
ax.bar(profiles.index.astype(str),profiles.users)
ax.set(title='Users per cluster',xlabel='Cluster ID',ylabel='Users')
fig.tight_layout(); fig.savefig(OUT/'cluster_sizes.png'); plt.show()

## Cell 15: Seed stability

In [ ]:
stability=[]
with threadpool_limits(limits=2):
    for seed in [7,21,99]:
        alternative=KMeans(n_clusters=selected,n_init=10,random_state=seed).fit_predict(Z)
        stability.append({'seed':seed,'ARI':adjusted_rand_score(model.labels_,alternative)})
display(pd.DataFrame(stability))

## Cell 16: Save outputs and model

In [ ]:
df.to_csv(OUT/'users_with_clusters.csv',index=False)
profiles.to_csv(OUT/'cluster_profiles.csv')
centers.to_csv(OUT/'standardized_cluster_centers.csv')
pca_df.to_csv(OUT/'pca_coordinates.csv',index=False)
pd.DataFrame(stability).to_csv(OUT/'seed_stability.csv',index=False)
for c in profiles.index:
    df.loc[df.cluster_id==c,['user_id','cluster_id','segment']].to_csv(OUT/f'cluster_{c}_users.csv',index=False)
joblib.dump({'features':FEATURES,'preprocessor':prep,'model':model,'segment_names':names},OUT/'segmentation_model.joblib')
summary={'source':'SYNTHETIC DEMONSTRATION' if DATA_PATH is None else str(DATA_PATH),'users':len(df),'selected_k':selected,'silhouette_best_k':best,**evaluation.set_index('k').loc[selected].to_dict(),'pca_2d_variance':float(pca.explained_variance_ratio_.sum()),'seed_stability':stability}
(OUT/'run_summary.json').write_text(json.dumps(summary,indent=2))
(OUT/'project_report.txt').write_text(json.dumps(summary,indent=2)+'\n\n'+profiles.round(3).to_string()+'\n\nSynthetic results are not actual-dataset findings. Engagement is not a churn probability or monetary value. PCA is visualization only.')
print(json.dumps(summary,indent=2))

## Cell 17: New-user prediction

In [ ]:
def predict_new_users(data,model_path):
    bundle=joblib.load(model_path)  # Trusted model files only.
    if bundle['features']!=FEATURES: raise ValueError('Feature configuration mismatch')
    result=data.rename(columns=COLUMN_MAPPING).copy()
    scaled=bundle['preprocessor'].transform(clean_features(result))
    result['cluster_id']=bundle['model'].predict(scaled)
    result['segment']=result.cluster_id.map(bundle['segment_names'])
    return result
new_users=pd.DataFrame({'user_id':['NEW_001','NEW_002'],'sessions_per_week':[24,2],'avg_session_minutes':[30,5],'active_days_30':[25,3],'features_used':[10,2],'days_since_last_login':[1,24]})
predictions=predict_new_users(new_users,OUT/'segmentation_model.joblib')
predictions.to_csv(OUT/'new_user_predictions.csv',index=False)
display(predictions)

## Cell 18: ZIP all computed results

In [ ]:
import shutil
archive=shutil.make_archive(str(OUT),'zip',root_dir=str(OUT))
print('Created:',archive)

## Optional Google Colab download
Run the following only in Colab:
```python
from google.colab import files
files.download(archive)
```

Use one row per user and consistent time windows and units. Source dataset columns were not inspected. Synthetic data deliberately contains four patterns. Review real-data outliers, feature redundancy, temporal stability, and business outcomes. Heuristic engagement tiers are not validated churn predictions. Notebook cells are supplied ready to run, without claimed execution outputs.